In [ ]:
# ==============================
# Faster / safer bbox3 runner + rotation squeeze
# - No unary_union (much faster)
# - Optimize only big groups (configurable) instead of all 200 every time
# - Keep GLOBAL best by total score (auto-save best)
# - Optional periodic full pass
# ==============================

DEBUG = False          # True -> run only 1 (r,n) pair, for quick sanity
MAX_HOURS = 11.7      # overall wallclock budget
BBOX_TIMEOUT_SEC = 1200  # per bbox3 run timeout (20min)

# ---- rotation squeeze controls ----
MIN_GROUP_TO_SQUEEZE = 80       # only optimize groups n>=this (huge speedup)
FULL_SWEEP_EVERY = 0            # 0=disable. If >0: every K bbox3 runs do a full sweep (n=200..3)
EPS_IMPROVE = 1e-10             # accept tiny improvements

# ---- bbox3 parameter grid (same idea as your code) ----
N_MIN, N_MAX, N_STEP = 1000, 2000, 100
R_MIN, R_MAX, R_STEP = 10, 90, 10

# ---- IO ----
from shutil import copy
import os, time, subprocess
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

copy('/kaggle/input/intergration-of-existing-result-current-best/submission.csv', '/kaggle/working/submission.csv')
copy('/kaggle/input/santa-submission/bbox3', '/kaggle/working/bbox3')
os.chmod('/kaggle/working/bbox3', 0o755)

from decimal import Decimal, getcontext
from shapely import affinity
from shapely.geometry import Polygon
from scipy.spatial import ConvexHull
from scipy.optimize import minimize_scalar

getcontext().prec = 30
scale_factor = 1  # keep as original

# =========================
# Tree shape (same vertices)
# =========================
class ChristmasTree:
    def __init__(self, center_x='0', center_y='0', angle='0'):
        self.center_x = Decimal(str(center_x))
        self.center_y = Decimal(str(center_y))
        self.angle = Decimal(str(angle))

        trunk_w = Decimal('0.15')
        trunk_h = Decimal('0.2')
        base_w  = Decimal('0.7')
        mid_w   = Decimal('0.4')
        top_w   = Decimal('0.25')
        tip_y   = Decimal('0.8')
        tier_1_y = Decimal('0.5')
        tier_2_y = Decimal('0.25')
        base_y = Decimal('0.0')
        trunk_bottom_y = -trunk_h

        poly = Polygon(
            [
                (Decimal('0.0') * scale_factor, tip_y * scale_factor),
                (top_w/Decimal('2')*scale_factor, tier_1_y*scale_factor),
                (top_w/Decimal('4')*scale_factor, tier_1_y*scale_factor),
                (mid_w/Decimal('2')*scale_factor, tier_2_y*scale_factor),
                (mid_w/Decimal('4')*scale_factor, tier_2_y*scale_factor),
                (base_w/Decimal('2')*scale_factor, base_y*scale_factor),
                (trunk_w/Decimal('2')*scale_factor, base_y*scale_factor),
                (trunk_w/Decimal('2')*scale_factor, trunk_bottom_y*scale_factor),
                (-(trunk_w/Decimal('2'))*scale_factor, trunk_bottom_y*scale_factor),
                (-(trunk_w/Decimal('2'))*scale_factor, base_y*scale_factor),
                (-(base_w/Decimal('2'))*scale_factor, base_y*scale_factor),
                (-(mid_w/Decimal('4'))*scale_factor, tier_2_y*scale_factor),
                (-(mid_w/Decimal('2'))*scale_factor, tier_2_y*scale_factor),
                (-(top_w/Decimal('4'))*scale_factor, tier_1_y*scale_factor),
                (-(top_w/Decimal('2'))*scale_factor, tier_1_y*scale_factor),
            ]
        )
        poly = affinity.rotate(poly, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(
            poly,
            xoff=float(self.center_x * scale_factor),
            yoff=float(self.center_y * scale_factor),
        )

    def clone(self):
        return ChristmasTree(str(self.center_x), str(self.center_y), str(self.angle))


# =========================
# FAST side length (NO unary_union)
# =========================
def get_tree_list_side_length_fast(tree_list):
    # compute union bounds by min/max of each polygon bounds (no union)
    # bounds = (minx, miny, maxx, maxy)
    minx = float("inf"); miny = float("inf")
    maxx = float("-inf"); maxy = float("-inf")
    for t in tree_list:
        bx0, by0, bx1, by1 = t.polygon.bounds
        if bx0 < minx: minx = bx0
        if by0 < miny: miny = by0
        if bx1 > maxx: maxx = bx1
        if by1 > maxy: maxy = by1
    return Decimal(max(maxx - minx, maxy - miny)) / Decimal(scale_factor)


def get_total_score(side_len_dict):
    score = Decimal("0")
    for k, v in side_len_dict.items():
        score += v * v / Decimal(k)
    return score


# =========================
# Parse submission -> per-group trees + side
# =========================
def parse_csv(csv_path):
    df = pd.read_csv(csv_path)
    df["x"] = df["x"].astype(str).str.strip().str.lstrip("sS")
    df["y"] = df["y"].astype(str).str.strip().str.lstrip("sS")
    df["deg"] = df["deg"].astype(str).str.strip().str.lstrip("sS")
    df[["group_id", "item_id"]] = df["id"].astype(str).str.split("_", n=1, expand=True)

    dict_of_tree_list = {}
    dict_of_side_len = {}

    for gid, g in df.groupby("group_id", sort=False):
        trees = [ChristmasTree(row["x"], row["y"], row["deg"]) for _, row in g.iterrows()]
        dict_of_tree_list[gid] = trees
        dict_of_side_len[gid] = get_tree_list_side_length_fast(trees)

    return df, dict_of_tree_list, dict_of_side_len


# =========================
# Rotation optimization on hull points (fast)
# =========================
def calculate_bbox_side_at_angle(angle_deg, points_xy):
    a = np.deg2rad(angle_deg)
    c, s = np.cos(a), np.sin(a)
    # rotate points by angle: p' = p @ R^T where R = [[c,-s],[s,c]]
    # using R^T = [[c,s],[-s,c]]
    rotT = np.array([[c, s], [-s, c]], dtype=np.float64)
    rp = points_xy @ rotT
    mn = rp.min(axis=0)
    mx = rp.max(axis=0)
    return max(mx[0]-mn[0], mx[1]-mn[1])


def optimize_rotation_for_group(trees):
    # gather all exterior points
    all_pts = []
    for t in trees:
        all_pts.extend(list(t.polygon.exterior.coords))
    pts = np.asarray(all_pts, dtype=np.float64)
    if pts.shape[0] < 3:
        # degenerate, don't rotate
        side0 = calculate_bbox_side_at_angle(0.0, pts)
        return Decimal(side0) / Decimal(scale_factor), 0.0

    # convex hull (reduce points)
    hull = ConvexHull(pts)
    hull_pts = pts[hull.vertices]

    side0 = calculate_bbox_side_at_angle(0.0, hull_pts)

    res = minimize_scalar(
        lambda a: calculate_bbox_side_at_angle(a, hull_pts),
        bounds=(0.0, 90.0),
        method="bounded"
    )
    side_best = float(res.fun)
    ang_best = float(res.x)

    if side0 - side_best > 1e-12:
        return Decimal(side_best) / Decimal(scale_factor), ang_best
    else:
        return Decimal(side0) / Decimal(scale_factor), 0.0


def apply_group_rotation(trees, angle_deg):
    if not trees or abs(angle_deg) < 1e-12:
        return [t.clone() for t in trees]

    # rotation center = center of current bounds
    minx = float("inf"); miny = float("inf")
    maxx = float("-inf"); maxy = float("-inf")
    for t in trees:
        bx0, by0, bx1, by1 = t.polygon.bounds
        if bx0 < minx: minx = bx0
        if by0 < miny: miny = by0
        if bx1 > maxx: maxx = bx1
        if by1 > maxy: maxy = by1
    center = np.array([(minx+maxx)/2.0, (miny+maxy)/2.0], dtype=np.float64)

    a = np.deg2rad(angle_deg)
    c, s = np.cos(a), np.sin(a)
    R = np.array([[c, -s], [s, c]], dtype=np.float64)

    pts = np.array([[float(t.center_x), float(t.center_y)] for t in trees], dtype=np.float64)
    rot = (pts - center) @ R.T + center

    out = []
    for i, t in enumerate(trees):
        out.append(
            ChristmasTree(
                Decimal(rot[i, 0]),
                Decimal(rot[i, 1]),
                Decimal(t.angle + Decimal(str(angle_deg)))
            )
        )
    return out


# =========================
# Squeeze by rotating groups (selective)
# =========================
def squeeze_by_rotation(in_csv="submission.csv", out_csv="submission.csv", min_group=80):
    df, tree_dict, side_dict = parse_csv(in_csv)
    score0 = get_total_score(side_dict)

    improved_groups = 0
    # process large groups first
    for n in range(200, 2, -1):
        if n < min_group:
            break
        gid = f"{n:03d}"
        if gid not in tree_dict:
            continue
        trees = tree_dict[gid]
        cur_side = side_dict[gid]

        best_side, best_ang = optimize_rotation_for_group(trees)
        if best_side + Decimal(str(EPS_IMPROVE)) < cur_side:
            tree_dict[gid] = apply_group_rotation(trees, best_ang)
            side_dict[gid] = best_side
            improved_groups += 1

    score1 = get_total_score(side_dict)
    if score1 + Decimal(str(EPS_IMPROVE)) < score0:
        # write back
        rows = []
        for gid, trees in tree_dict.items():
            for i, t in enumerate(trees):
                rows.append({
                    "id": f"{gid}_{i}",
                    "x": f"s{t.center_x}",
                    "y": f"s{t.center_y}",
                    "deg": f"s{t.angle}",
                })
        pd.DataFrame(rows).to_csv(out_csv, index=False)

    return float(score0), float(score1), improved_groups


def squeeze_full(in_csv="submission.csv", out_csv="submission.csv"):
    # full sweep n=200..3 (slow)
    df, tree_dict, side_dict = parse_csv(in_csv)
    score0 = get_total_score(side_dict)

    improved_groups = 0
    for n in range(200, 2, -1):
        gid = f"{n:03d}"
        if gid not in tree_dict:
            continue
        trees = tree_dict[gid]
        cur_side = side_dict[gid]
        best_side, best_ang = optimize_rotation_for_group(trees)
        if best_side + Decimal(str(EPS_IMPROVE)) < cur_side:
            tree_dict[gid] = apply_group_rotation(trees, best_ang)
            side_dict[gid] = best_side
            improved_groups += 1

    score1 = get_total_score(side_dict)
    if score1 + Decimal(str(EPS_IMPROVE)) < score0:
        rows = []
        for gid, trees in tree_dict.items():
            for i, t in enumerate(trees):
                rows.append({
                    "id": f"{gid}_{i}",
                    "x": f"s{t.center_x}",
                    "y": f"s{t.center_y}",
                    "deg": f"s{t.angle}",
                })
        pd.DataFrame(rows).to_csv(out_csv, index=False)

    return float(score0), float(score1), improved_groups


# =========================
# Main experiment loop
# =========================
def run_bbox_with_timeout(debug=False):
    os.makedirs("bbox_sub", exist_ok=True)
    log_file = "bbox_experiments.log"

    start_time = datetime.now()
    deadline = start_time + timedelta(hours=MAX_HOURS)
    print(f"Start: {start_time} | deadline: {deadline} | budget: {MAX_HOURS} hours")

    n_values = list(range(N_MIN, N_MAX + 1, N_STEP))
    r_values = list(range(R_MIN, R_MAX + 1, R_STEP))
    total_runs = len(n_values) * len(r_values)

    # initial squeeze once
    s0, s1, ig = squeeze_by_rotation("submission.csv", "submission.csv", min_group=MIN_GROUP_TO_SQUEEZE)
    print(f"[init squeeze] {s0:.12f} -> {s1:.12f} | improved_groups={ig}")

    # track global best
    best_score = s1
    best_path = "/kaggle/working/submission_best.csv"
    copy("submission.csv", best_path)

    completed = 0
    bbox_runs = 0

    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"\n{'='*60}\nStart: {start_time}\nBudget hours: {MAX_HOURS}\n{'='*60}\n")

        for r in r_values:
            for n in n_values:
                now = datetime.now()
                if now >= deadline:
                    print(f"⏰ Time up at {now}. Stopping.")
                    f.write(f"\n⏰ Time up at {now}. Stopping.\n")
                    # restore best
                    copy(best_path, "submission.csv")
                    print(f"✅ Restored GLOBAL BEST: {best_score:.12f} -> submission.csv")
                    return

                completed += 1
                bbox_runs += 1
                progress = 100.0 * completed / total_runs
                elapsed = now - start_time
                print(f"\n[{progress:5.1f}%] run={completed}/{total_runs} | elapsed={elapsed} | n={n} r={r}")

                # run bbox3 (in-place on submission.csv)
                try:
                    res = subprocess.run(
                        ["/kaggle/working/bbox3", "-n", str(n), "-r", str(r)],
                        cwd="/kaggle/working",
                        capture_output=True,
                        text=True,
                        timeout=BBOX_TIMEOUT_SEC
                    )
                    if res.stderr:
                        print("bbox3 stderr:", res.stderr[:3000])
                except subprocess.TimeoutExpired:
                    print(f"⚠ bbox3 timeout for n={n} r={r} (>{BBOX_TIMEOUT_SEC}s). Skipping.")
                    f.write(f"bbox3 timeout: n={n} r={r}\n")
                    if debug:
                        copy(best_path, "submission.csv")
                        return
                    continue
                except Exception as e:
                    print("❌ bbox3 error:", e)
                    f.write(f"bbox3 error: {e}\n")
                    if debug:
                        copy(best_path, "submission.csv")
                        return
                    continue

                # backup the raw bbox3 output
                snap = f"bbox_sub/submi-n{n}_r{r}_i{completed}.csv"
                try:
                    copy("submission.csv", snap)
                except Exception as e:
                    print("⚠ snapshot save failed:", e)

                # squeeze by rotation on large groups (fast)
                before, after, ig = squeeze_by_rotation("submission.csv", "submission.csv", min_group=MIN_GROUP_TO_SQUEEZE)
                print(f"[squeeze>= {MIN_GROUP_TO_SQUEEZE}] {before:.12f} -> {after:.12f} | improved_groups={ig}")

                # optional full sweep occasionally
                if FULL_SWEEP_EVERY and (bbox_runs % FULL_SWEEP_EVERY == 0):
                    b2, a2, ig2 = squeeze_full("submission.csv", "submission.csv")
                    print(f"[FULL squeeze] {b2:.12f} -> {a2:.12f} | improved_groups={ig2}")
                    after = a2

                # update global best
                if after + 1e-12 < best_score:
                    best_score = after
                    copy("submission.csv", best_path)
                    print(f"🏆 GLOBAL BEST updated: {best_score:.12f} (saved to {best_path})")

                # log
                f.write(f"[{now}] n={n} r={r} squeeze_score={after:.12f} best={best_score:.12f}\n")
                f.flush()

                if debug:
                    print("DEBUG=True -> stopping after first run.")
                    copy(best_path, "submission.csv")
                    print(f"✅ Restored GLOBAL BEST: {best_score:.12f} -> submission.csv")
                    return

        # finished all grid
        copy(best_path, "submission.csv")
        print(f"\n✅ Grid done. GLOBAL BEST: {best_score:.12f} -> submission.csv")
        f.write(f"\n✅ Done. GLOBAL BEST: {best_score:.12f}\n")


run_bbox_with_timeout(debug=DEBUG)
